# Receiver Clock Bias and Drift Simulation

This notebook generates the stochastic "True" clock bias and drift for the receiver over the course of the simulation. It utilizes a 2-state random walk model based on the Allan variance parameters of the clock.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from orbit_determination.clock_sim import ClockSimulator
from orbit_determination.time_utils import julian_to_datetime

In [ ]:
# Define paths
project_root = Path.cwd().parent
data_path = project_root / 'data'

# Load the reference receiver file to extract the exact simulation timeline
rcver_path = data_path / 'raw' / 'receiver_orbits' / 'eor' / 'sample2.txt'

# Load Julian dates and convert them to elapsed seconds
jd_array = np.loadtxt(rcver_path, skiprows=1, usecols=(0,))
dates_utc = np.array([julian_to_datetime(jd) for jd in jd_array])
t_array = np.array([(date - dates_utc[0]).total_seconds() for date in dates_utc])

print(f"Loaded {len(t_array)} time steps.")

In [ ]:
# Initialize simulator with standard OCXO Allan variance parameters
simulator = ClockSimulator()

# Run the simulation
clock_bias, clock_drift = simulator.simulate(t_array, init_bias=0.0, init_drift=0.0)

# Export the results to the interim data folder
export_dir = data_path / 'interim' / 'clock_sim'
export_dir.mkdir(parents=True, exist_ok=True)

# Stack arrays into a single (2, N) array and save
output_data = np.vstack((clock_bias, clock_drift))
np.save(export_dir / 'true_clock_bias_drift.npy', output_data)

print(f"Successfully exported clock states to {export_dir}")

In [ ]:
# Plot the generated bias to ensure it looks like a proper random walk
plt.figure(figsize=(10, 4))
plt.plot(t_array / 3600, clock_bias, label="Clock Bias")
plt.xlabel("Time [hours]")
plt.ylabel("Bias [m]")
plt.title("Simulated Receiver Clock Bias")
plt.grid(True, linestyle=":", alpha=0.7)
plt.legend()
plt.show()
